In [6]:
import geemap
import utils as F
import ee
import concurrent.futures
import glob
import rasterio
import geopandas as gpd
from rasterio.merge import merge
from shapely.geometry import mapping
from shapely.ops import unary_union, transform
import shapely
import concurrent.futures
import os
import ee
ee.Initialize(project="ee-103dg017")
ee.Authenticate()

True

In [7]:
def force_2d(geom):
    # 兼容 Shapely 1.x/2.x：把 (x,y,z) -> (x,y)
    return transform(lambda x, y, z=None: (x, y), geom)

def shapely_to_ee_geometry(shp_geom):
    # 1) 先强制 2D
    shp_geom = force_2d(shp_geom)
    # 2) 修复无效几何（自相交等）
    if not shp_geom.is_valid:
        shp_geom = shp_geom.buffer(0)
    geo_json = mapping(shp_geom)
    gtype = geo_json["type"]
    coords = geo_json["coordinates"]

    if gtype == "Polygon":
        return ee.Geometry.Polygon(coords)
    elif gtype == "MultiPolygon":
        return ee.Geometry.MultiPolygon(coords)
    else:
        raise ValueError(f"Unsupported geometry type: {gtype}")
    
def cal_cloud_per(img):
    
    cloud_img = img.select('cloud')
    Cloud_count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.sum(),
      geometry = roi,
      scale = 30,
      maxPixels = 1e13
    ).get('cloud')
  
    count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.count(),
      geometry = roi,
      scale = 30,
      maxPixels =  1e13
    ).get('cloud')
    
    cloud_per = ee.Number(Cloud_count).divide(count).multiply(100)
    
    return img.set('cloud_per', cloud_per)
  
def CLIP(img):
    return img.clip(roi)

In [116]:
target_region = 'Severn_Esturay'
year = 2020

start_date, end_date = f'{year}-01-01', f'{year+1}-01-01'

# --- 选 Morecambe 区 ---
path = r"E:\GEEdownload\UK_TF_roi\UK_FK.shp"
gdf = gpd.read_file(path)
region_grid = gdf[(gdf['name'] == target_region)]
MGRS_TILE = region_grid['MGRS_TILE'].values[0]
print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

if region_grid.empty:
    raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")
merged_ = unary_union([force_2d(geom) for geom in region_grid.geometry if geom is not None])
roi = shapely_to_ee_geometry(merged_)


 5 MGRS_tile were exit!  
 MGRS_list: False


In [15]:
target_region = 'Henber'
year = 2019

start_date, end_date = f'{year}-01-01', f'{year+1}-01-01'

# --- 选 Morecambe 区 ---
path = r"E:\GEEdownload\UK_TF_roi\UK_FK.shp"
gdf = gpd.read_file(path)
region_grid = gdf[(gdf['name'] == target_region)]
MGRS_TILE = region_grid['MGRS_TILE'].values[0]
# print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

if region_grid.empty:
    raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")
merged_ = unary_union([force_2d(geom) for geom in region_grid.geometry if geom is not None])
roi = shapely_to_ee_geometry(merged_)

cloud_limit = 40

S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 100)

if MGRS_TILE != 'False':
    print('--apply MGRS_TILE--', MGRS_TILE)
    S2 = S2.filter(ee.Filter.eq('MGRS_TILE', MGRS_TILE))
            
S2 = S2.map(F.VI_cal_S2).map(F.add_cloud_bands)\
    .map(CLIP).map(cal_cloud_per)\
    .filter(ee.Filter.gt('cloud_per', cloud_limit))
    
L9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
L8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
L9 = F.Landsat_processing(roi, L9_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)
L8 = F.Landsat_processing(roi, L8_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)

S2_sum_TFI = S2.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
S2_sum_cloud = S2.select('cloud').toBands().reduce(ee.Reducer.sum())
S2_NEF = S2_sum_TFI.divide(S2_sum_cloud).rename('S2_NEF')

L89_collect = L8.merge(L9)
L89_collect = ee.ImageCollection(L89_collect)
L89_sum_TFI = L89_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
L89_sum_cloud = L89_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
L89_NEF =  L89_sum_TFI.divide(L89_sum_cloud).rename('L89_NEF')

all_TF_collect = S2.merge(L8).merge(L9)
all_TF_collect = ee.ImageCollection(all_TF_collect)
sum_TF = all_TF_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
sum_cloud = all_TF_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
Frequency = sum_TF.divide(sum_cloud).rename('NEF')
            
Feature_bands = Frequency.addBands(S2_NEF).addBands(L89_NEF)
filename = f'{target_region}_{year}_NEF'
            
task = ee.batch.Export.image.toDrive(
    image=Feature_bands.toFloat(),
    description= filename,
    folder='FB_FIG4',
    fileNamePrefix=filename,
    region=roi,
    scale=30,
    maxPixels=1e13
    )
task.start()
print(f"\n✅ {filename}已提交，请到 https://code.earthengine.google.com/tasks 查看进度")



✅ Henber_2019_NEF已提交，请到 https://code.earthengine.google.com/tasks 查看进度


In [17]:
target_region = 'Henber'
year = 2023

start_date, end_date = f'{year}-01-01', f'{year+1}-01-01'

# --- 选 Morecambe 区 ---
path = r"E:\GEEdownload\UK_TF_roi\UK_FK.shp"
gdf = gpd.read_file(path)
region_grid = gdf[(gdf['name'] == target_region)]
MGRS_TILE = region_grid['MGRS_TILE'].values[0]
# print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

if region_grid.empty:
    raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")
merged_ = unary_union([force_2d(geom) for geom in region_grid.geometry if geom is not None])
roi = shapely_to_ee_geometry(merged_)


S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 20)
# S2 = S2.filter(ee.Filter.eq('MGRS_TILE', MGRS_TILE))
S2 = S2.map(F.Cal_doy)
#     .map(CLIP).map(cal_cloud_per)\
#     .filter(ee.Filter.gt('cloud_per', cloud_limit))
    

M = geemap.Map()

In [ ]:
doy_list = S2.aggregate_array('DOY').distinct()
doy_list

In [ ]:

viz = {'bands':['B8', 'B4', 'B3'], 'min':0, 'max':4000}
unique_doy = S2.filter(ee.Filter.eq('DOY', 321)).mosaic().clip(roi)
# kk = unique_doy.clip(roi)
M.addLayer(unique_doy, viz, 'kk')
M


In [20]:
filename = f'Henber_2023_321'
            
task = ee.batch.Export.image.toDrive(
    image=unique_doy.select(['B2', 'B3', 'B4', 'B8']).toFloat(),
    description= filename,
    folder='FB_FIG4',
    fileNamePrefix=filename,
    region=roi,
    scale=30,
    maxPixels=1e13
)
task.start()

In [18]:
M = geemap.Map()
M.addLayer(roi)
M.centerObject(roi, 10)
M

Map(center=[54.19983266640465, -3.204241353773345], controls=(WidgetControl(options=['position', 'transparent_…

In [17]:
# FB_importance image download
target_region = 'Morecambe'
year = 2023

start_date, end_date = f'{year}-01-01', f'{year+1}-01-01'

# --- 选 Morecambe 区 ---
path = r"E:\GEEdownload\UK_TF_roi\UK_FK.shp"
gdf = gpd.read_file(path)
region_grid = gdf[(gdf['name'] == target_region)]
MGRS_TILE = region_grid['MGRS_TILE'].values[0]
# print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

if region_grid.empty:
    raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")
merged_ = unary_union([force_2d(geom) for geom in region_grid.geometry if geom is not None])
roi = shapely_to_ee_geometry(merged_)

cloud_limit = 40

S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 100)

if MGRS_TILE != 'False':
    print('--apply MGRS_TILE--', MGRS_TILE)
    S2 = S2.filter(ee.Filter.eq('MGRS_TILE', MGRS_TILE))
            
S2 = S2.map(F.VI_cal_S2).map(F.add_cloud_bands)\
    .map(CLIP).map(cal_cloud_per)\
    .filter(ee.Filter.gt('cloud_per', cloud_limit))
    
L9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
L8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
L9 = F.Landsat_processing(roi, L9_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)
L8 = F.Landsat_processing(roi, L8_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)

S2_sum_TFI = S2.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
S2_sum_cloud = S2.select('cloud').toBands().reduce(ee.Reducer.sum())
S2_NEF = S2_sum_TFI.divide(S2_sum_cloud).rename('NEF_S2')

S2_2022 = S2.map(F.normalized_TFI).map(F.TF_get)
Norm_S2_2022 = S2_2022.map(F.Norm)
cloud_sum = S2_2022.select('cloud').toBands().reduce(ee.Reducer.sum())

NDWI_norm_min = S2_2022.filter(ee.Filter.gt('cloud_per', 60)).select('NDWI_norm').reduce(ee.Reducer.percentile([1])).rename('NDWI_min_S2')
S2_MNDWI_min = S2_2022.select('MNDWI').reduce(ee.Reducer.percentile([2])).rename('MNDWI_min_S2')
NDWI_mean = S2_2022.select('NDWI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('NDWI_mean_S2')
MNDWI_mean = S2_2022.select('MNDWI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('MNDWI_mean_S2')
mean_B1 = S2_2022.select('B1').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B1_mean_S2')
mean_B2 = S2_2022.select('B2').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B2_mean_S2')
mean_B3 = S2_2022.select('B3').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B3_mean_S2')
mean_B4 = S2_2022.select('B4').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B4_mean_S2')
mean_B5 = S2_2022.select('B5').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B5_mean_S2')
mean_B6 = S2_2022.select('B6').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B6_mean_S2')
mean_B7 = S2_2022.select('B7').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B7_mean_S2')
mean_B8 = S2_2022.select('B8').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B8_mean_S2')
mean_B8A = S2_2022.select('B8A').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B8A_mean_S2')
mean_B11 = S2_2022.select('B11').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B11_mean_S2')
mean_B12 = S2_2022.select('B12').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('B12_mean_S2')

L89_collect = L8.merge(L9)
L89_collect = ee.ImageCollection(L89_collect)
L89_sum_TFI = L89_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
L89_sum_cloud = L89_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
L89_NEF =  L89_sum_TFI.divide(L89_sum_cloud).rename('NEF_L89')

NDWI_mean_L8 = L89_collect.select('NDWI').toBands().reduce(ee.Reducer.sum()).divide(L89_sum_cloud).rename('NDWI_mean_L8')
MNDWI_mean_L8 = L89_collect.select('MNDWI').toBands().reduce(ee.Reducer.sum()).divide(L89_sum_cloud).rename('MNDWI_mean_L8')
mean_SR_B1 = L89_collect.select('SR_B1').toBands().reduce(ee.Reducer.sum()).divide(L89_sum_cloud).rename('B1_mean_L89')
mean_SR_B2 = L89_collect.select('SR_B2').toBands().reduce(ee.Reducer.sum()).divide(L89_sum_cloud).rename('B2_mean_L89')
mean_SR_B3 = L89_collect.select('SR_B3').toBands().reduce(ee.Reducer.sum()).divide(L89_sum_cloud).rename('B3_mean_L89')
mean_SR_B4 = L89_collect.select('SR_B4').toBands().reduce(ee.Reducer.sum()).divide(L89_sum_cloud).rename('B4_mean_L89')
mean_SR_B5 = L89_collect.select('SR_B5').toBands().reduce(ee.Reducer.sum()).divide(L89_sum_cloud).rename('B5_mean_L89')
mean_SR_B6 = L89_collect.select('SR_B6').toBands().reduce(ee.Reducer.sum()).divide(L89_sum_cloud).rename('B6_mean_L89')
mean_SR_B7 = L89_collect.select('SR_B7').toBands().reduce(ee.Reducer.sum()).divide(L89_sum_cloud).rename('B7_mean_L89')


all_TF_collect = S2.merge(L8).merge(L9)
all_TF_collect = ee.ImageCollection(all_TF_collect)
sum_TF = all_TF_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
sum_cloud = all_TF_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
Frequency = sum_TF.divide(sum_cloud).rename('NEF_S2_L89')
            
Feature_bands = Frequency.addBands(S2_NEF).addBands(L89_NEF)\
                         .addBands(mean_B1).addBands(mean_B2).addBands(mean_B3).addBands(mean_B4).addBands(mean_B5).addBands(mean_B6)\
                         .addBands(mean_B7).addBands(mean_B8).addBands(mean_B8A).addBands(mean_B11).addBands(mean_B12)\
                         .addBands(NDWI_mean).addBands(MNDWI_mean).addBands(NDWI_norm_min).addBands(S2_MNDWI_min)\
                         .addBands(mean_SR_B1).addBands(mean_SR_B2).addBands(mean_SR_B3).addBands(mean_SR_B4).addBands(mean_SR_B5)\
                         .addBands(mean_SR_B5).addBands(mean_SR_B6).addBands(mean_SR_B7)\
                         .addBands(NDWI_mean_L8).addBands(MNDWI_mean_L8)


filename = f'{target_region}_{year}_importance_FB'

task = ee.batch.Export.image.toDrive(
    image=Feature_bands.toFloat(),
    description= filename,
    folder='FB_FIG4',
    fileNamePrefix=filename,
    region=roi,
    scale=30,
    maxPixels=1e13
    )
task.start()

In [ ]:
#----------------for single region---------------------
target_region = 'Wash_bay'
year = 2022

img_num = 40
img_num2 = 40
seed = 1
start_date, end_date = f'{year}-01-01', f'{year+1}-01-01'

# --- 选 Morecambe 区 ---
path = r"E:\GEEdownload\UK_TF_roi\UK_FK.shp"
gdf = gpd.read_file(path)
region_grid = gdf[(gdf['name'] == target_region)]
MGRS_TILE = region_grid['MGRS_TILE'].values[0]
# print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

if region_grid.empty:
    raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")
merged_ = unary_union([force_2d(geom) for geom in region_grid.geometry if geom is not None])
roi = shapely_to_ee_geometry(merged_)


edge_path = f'E:\GEEdownload\TK-TF_edge_collect\{target_region}_edge.shp'
edge_gdf = gpd.read_file(edge_path)
edge_gdf  = geemap.geopandas_to_ee(edge_gdf)       
edge_roi = edge_gdf.geometry() 

cloud_limit = 40


S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 100)

if MGRS_TILE != 'False':
    print('--apply MGRS_TILE--', MGRS_TILE)
    S2 = S2.filter(ee.Filter.eq('MGRS_TILE', MGRS_TILE))
            
S2 = S2.map(F.VI_cal_S2).map(F.add_cloud_bands)\
    .map(CLIP).map(cal_cloud_per)\
    .filter(ee.Filter.gt('cloud_per', cloud_limit))

L9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
L8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
L9 = F.Landsat_processing(roi, L9_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)
L8 = F.Landsat_processing(roi, L8_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)


S2 = S2.randomColumn('random', seed).sort('random')
S2 = ee.ImageCollection(S2)
S2_ = S2.map(F.normalized_TFI).map(F.TF_get)


S2_TF = S2_.map(F.TF_binary_cal).toBands().select(ee.List.sequence(0, img_num2-1))
S2_cloud = S2_.select('cloud').toBands().select(ee.List.sequence(0, img_num2-1))

S2_sum_TF = S2_TF.reduce(ee.Reducer.sum())
S2_cloud_sum = S2_cloud.reduce(ee.Reducer.sum())

NEF = S2_sum_TF.divide(S2_cloud_sum).rename('NEF')
NDWI_mean = S2_.select('NDWI').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).rename('mean_NDWI')
MNDWI_mean = S2_.select('MNDWI').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).rename('mean_MNDWI')
composit_b8A = S2_.select('B8A').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).divide(10000).rename('mean_B8A')


if img_num > img_num2:#加入 L8/L9 影像
    
    L89_num = img_num - img_num2
    L89_TF_collect = L8.merge(L9).randomColumn('random', seed).sort('random')
    L89_TF_collect = ee.ImageCollection(L89_TF_collect)
    L89_sum_TF = L89_TF_collect.map(F.TF_binary_cal).toBands().select(ee.List.sequence(0, L89_num-1))#.reduce(ee.Reducer.sum())
    L89_sum_cloud = L89_TF_collect.select('cloud').toBands().select(ee.List.sequence(0, L89_num-1))#.reduce(ee.Reducer.sum())
    
    sum_TF = L89_sum_TF.addBands(S2_TF).reduce(ee.Reducer.sum())
    sum_cloud = L89_sum_cloud.addBands(S2_cloud).reduce(ee.Reducer.sum())
    NEF = sum_TF.divide(sum_cloud).rename('NEF')
    
#-------------apply all images to construct mask and S2_NDWI_min---------------------

all_TF_collect = S2.merge(L8).merge(L9)
all_TF_collect = ee.ImageCollection(all_TF_collect)
sum_TF = all_TF_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
sum_cloud = all_TF_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
Frequency_all = sum_TF.divide(sum_cloud).rename('Freq')

S2_2022 = S2.map(F.normalized_TFI).map(F.TF_get)
S2_NDWI_min = S2_2022.select('NDWI').reduce(ee.Reducer.percentile([2])).rename('S2_NDWI_min')

TF_mask = S2_NDWI_min.lt(-0.1).And(Frequency_all.gt(0.05)).And(Frequency_all.lt(0.95)).clip(edge_roi).unmask(0).rename('mask')

#-------------------------------------------------------------------------------------    

Feature_band = NEF.addBands(NDWI_mean).addBands(MNDWI_mean).addBands(composit_b8A).addBands(S2_NDWI_min).addBands(TF_mask)      
            

filename = f'{target_region}_{year}_FB6_new9_{img_num}'
            
task = ee.batch.Export.image.toDrive(
    image=Feature_band.toFloat(),
    description= filename,
    folder='Discussion_51',
    fileNamePrefix=filename,
    region=roi,
    scale=30,
    maxPixels=1e13
    )
task.start()
print(f"\n✅ {filename}已提交，请到 https://code.earthengine.google.com/tasks 查看进度")

--apply MGRS_TILE-- 30UYD

✅ Wash_bay_2022_FB6_new9_40已提交，请到 https://code.earthengine.google.com/tasks 查看进度


In [17]:
#----------------for batch region---------------------
# target_region = 'Wash_bay'
# year = 2022
# img_num_list = [10, 20, 30, 40, 50, 60, 70, 80, 90, 93]
# img_num2_list = [10, 20, 30, 40, 50, 60, 68, 68, 68, 68]

target_region = 'Thames_esturay'
year = 2022
img_num_list = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
img_num2_list = [10, 20, 30, 40, 50, 60, 61, 61, 61, 61]
seed = 5


for i in range(len(img_num_list)):
    
    img_num  = img_num_list[i]
    img_num2 = img_num2_list[i]


    start_date, end_date = f'{year}-01-01', f'{year+1}-01-01'

    # --- 选 Morecambe 区 ---
    path = r"E:\GEEdownload\UK_TF_roi\UK_FK.shp"
    gdf = gpd.read_file(path)
    region_grid = gdf[(gdf['name'] == target_region)]
    MGRS_TILE = region_grid['MGRS_TILE'].values[0]
    # print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

    if region_grid.empty:
        raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")
    merged_ = unary_union([force_2d(geom) for geom in region_grid.geometry if geom is not None])
    roi = shapely_to_ee_geometry(merged_)


    edge_path = f'E:\GEEdownload\TK-TF_edge_collect\{target_region}_edge.shp'
    edge_gdf = gpd.read_file(edge_path)
    edge_gdf  = geemap.geopandas_to_ee(edge_gdf)       
    edge_roi = edge_gdf.geometry() 

    cloud_limit = 40


    S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 100)

    # if MGRS_TILE != 'False':
    #     print('--apply MGRS_TILE--', MGRS_TILE)

    S2 = S2.filter(ee.Filter.eq('MGRS_TILE', '30UYC'))
      
    S2 = S2.map(F.VI_cal_S2).map(F.add_cloud_bands)\
        .map(CLIP).map(cal_cloud_per)\
        .filter(ee.Filter.gt('cloud_per', cloud_limit))

    L9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    L8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    L9 = F.Landsat_processing(roi, L9_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)
    L8 = F.Landsat_processing(roi, L8_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)


    S2 = S2.randomColumn('random', seed).sort('random')
    S2 = ee.ImageCollection(S2)
    S2_ = S2.map(F.normalized_TFI).map(F.TF_get)


    S2_TF = S2_.map(F.TF_binary_cal).toBands().select(ee.List.sequence(0, img_num2-1))
    S2_cloud = S2_.select('cloud').toBands().select(ee.List.sequence(0, img_num2-1))

    S2_sum_TF = S2_TF.reduce(ee.Reducer.sum())
    S2_cloud_sum = S2_cloud.reduce(ee.Reducer.sum())

    NEF = S2_sum_TF.divide(S2_cloud_sum).rename('NEF')
    NDWI_mean = S2_.select('NDWI').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).rename('mean_NDWI')
    MNDWI_mean = S2_.select('MNDWI').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).rename('mean_MNDWI')
    composit_b8A = S2_.select('B8A').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).divide(10000).rename('mean_B8A')


    if img_num > img_num2:#加入 L8/L9 影像
        
        L89_num = img_num - img_num2
        L89_TF_collect = L8.merge(L9).randomColumn('random', seed).sort('random')
        L89_TF_collect = ee.ImageCollection(L89_TF_collect)
        L89_sum_TF = L89_TF_collect.map(F.TF_binary_cal).toBands().select(ee.List.sequence(0, L89_num-1))#.reduce(ee.Reducer.sum())
        L89_sum_cloud = L89_TF_collect.select('cloud').toBands().select(ee.List.sequence(0, L89_num-1))#.reduce(ee.Reducer.sum())
        
        sum_TF = L89_sum_TF.addBands(S2_TF).reduce(ee.Reducer.sum())
        sum_cloud = L89_sum_cloud.addBands(S2_cloud).reduce(ee.Reducer.sum())
        NEF = sum_TF.divide(sum_cloud).rename('NEF')
        
    #-------------apply all images to construct mask and S2_NDWI_min---------------------

    all_TF_collect = S2.merge(L8).merge(L9)
    all_TF_collect = ee.ImageCollection(all_TF_collect)
    sum_TF = all_TF_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
    sum_cloud = all_TF_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
    Frequency_all = sum_TF.divide(sum_cloud).rename('Freq')

    S2_2022 = S2.map(F.normalized_TFI).map(F.TF_get)
    S2_NDWI_min = S2_2022.select('NDWI').reduce(ee.Reducer.percentile([2])).rename('S2_NDWI_min')

    TF_mask = S2_NDWI_min.lt(-0.1).And(Frequency_all.gt(0.05)).And(Frequency_all.lt(0.95)).clip(edge_roi).unmask(0).rename('mask')

    #-------------------------------------------------------------------------------------    

    Feature_band = NEF.addBands(NDWI_mean).addBands(MNDWI_mean).addBands(composit_b8A).addBands(S2_NDWI_min).addBands(TF_mask)      
                

    filename = f'{target_region}_{year}_FB6_new9_{img_num}_{seed}'
                
    task = ee.batch.Export.image.toDrive(
        image=Feature_band.toFloat(),
        description= filename,
        folder='Discussion_51',
        fileNamePrefix=filename,
        region=roi,
        scale=30,
        maxPixels=1e13
        )
    task.start()
    print(f"\n✅ {filename}已提交，请到 https://code.earthengine.google.com/tasks 查看进度")


✅ Thames_esturay_2022_FB6_new9_10_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Thames_esturay_2022_FB6_new9_20_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Thames_esturay_2022_FB6_new9_30_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Thames_esturay_2022_FB6_new9_40_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Thames_esturay_2022_FB6_new9_50_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Thames_esturay_2022_FB6_new9_60_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Thames_esturay_2022_FB6_new9_70_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Thames_esturay_2022_FB6_new9_80_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Thames_esturay_2022_FB6_new9_90_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Thames_esturay_2022_FB6_new9_100_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度


In [12]:
roi_path = 'G:\ICESat-2\Region4_map\Ribble1_map\Ribble1_roi.shp'
roi_gdf = gpd.read_file(roi_path)
roi  = geemap.geopandas_to_ee(roi_gdf)       
roi = roi.geometry() 

#----------------for batch region---------------------
# target_region = 'Wash_bay'
# year = 2022
# img_num_list = [10, 20, 30, 40, 50, 60, 70, 80, 90, 93]
# img_num2_list = [10, 20, 30, 40, 50, 60, 68, 68, 68, 68]

target_region = 'Ribble'
year = 2020
img_num_list = [10, 20, 30, 40, 50, 60, 70, 79]
img_num2_list = [10, 20, 30, 40, 50, 60, 66, 66]
seed = 5


for i in range(len(img_num_list)):
    
    img_num  = img_num_list[i]
    img_num2 = img_num2_list[i]


    start_date, end_date = f'{year}-01-01', f'{year+1}-01-01'

    # --- 选 Morecambe 区 ---
    # path = r"E:\GEEdownload\UK_TF_roi\UK_FK.shp"
    # gdf = gpd.read_file(path)
    # region_grid = gdf[(gdf['name'] == target_region)]
    # MGRS_TILE = region_grid['MGRS_TILE'].values[0]
    # print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

    # if region_grid.empty:
    #     raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")
    # merged_ = unary_union([force_2d(geom) for geom in region_grid.geometry if geom is not None])
    roi = roi


    edge_path = f'E:\GEEdownload\TK-TF_edge_collect\{target_region}_edge.shp'
    edge_gdf = gpd.read_file(edge_path)
    edge_gdf  = geemap.geopandas_to_ee(edge_gdf)       
    edge_roi = edge_gdf.geometry() 

    cloud_limit = 40


    S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 100)

    # if MGRS_TILE != 'False':
    #     print('--apply MGRS_TILE--', MGRS_TILE)

    S2 = S2.filter(ee.Filter.eq('MGRS_TILE', '30UVE'))
      
    S2 = S2.map(F.VI_cal_S2).map(F.add_cloud_bands)\
        .map(CLIP).map(cal_cloud_per)\
        .filter(ee.Filter.gt('cloud_per', cloud_limit))

    L9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    L8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    L9 = F.Landsat_processing(roi, L9_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)
    L8 = F.Landsat_processing(roi, L8_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)


    S2 = S2.randomColumn('random', seed).sort('random')
    S2 = ee.ImageCollection(S2)
    S2_ = S2.map(F.normalized_TFI).map(F.TF_get)


    S2_TF = S2_.map(F.TF_binary_cal).toBands().select(ee.List.sequence(0, img_num2-1))
    S2_cloud = S2_.select('cloud').toBands().select(ee.List.sequence(0, img_num2-1))

    S2_sum_TF = S2_TF.reduce(ee.Reducer.sum())
    S2_cloud_sum = S2_cloud.reduce(ee.Reducer.sum())

    NEF = S2_sum_TF.divide(S2_cloud_sum).rename('NEF')
    NDWI_mean = S2_.select('NDWI').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).rename('mean_NDWI')
    MNDWI_mean = S2_.select('MNDWI').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).rename('mean_MNDWI')
    composit_b8A = S2_.select('B8A').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).divide(10000).rename('mean_B8A')


    if img_num > img_num2:#加入 L8/L9 影像
        
        L89_num = img_num - img_num2
        L89_TF_collect = L8.merge(L9).randomColumn('random', seed).sort('random')
        L89_TF_collect = ee.ImageCollection(L89_TF_collect)
        L89_sum_TF = L89_TF_collect.map(F.TF_binary_cal).toBands().select(ee.List.sequence(0, L89_num-1))#.reduce(ee.Reducer.sum())
        L89_sum_cloud = L89_TF_collect.select('cloud').toBands().select(ee.List.sequence(0, L89_num-1))#.reduce(ee.Reducer.sum())
        
        sum_TF = L89_sum_TF.addBands(S2_TF).reduce(ee.Reducer.sum())
        sum_cloud = L89_sum_cloud.addBands(S2_cloud).reduce(ee.Reducer.sum())
        NEF = sum_TF.divide(sum_cloud).rename('NEF')
        
    #-------------apply all images to construct mask and S2_NDWI_min---------------------

    all_TF_collect = S2.merge(L8).merge(L9)
    all_TF_collect = ee.ImageCollection(all_TF_collect)
    sum_TF = all_TF_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
    sum_cloud = all_TF_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
    Frequency_all = sum_TF.divide(sum_cloud).rename('Freq')

    S2_2022 = S2.map(F.normalized_TFI).map(F.TF_get)
    S2_NDWI_min = S2_2022.select('NDWI').reduce(ee.Reducer.percentile([2])).rename('S2_NDWI_min')

    TF_mask = S2_NDWI_min.lt(-0.1).And(Frequency_all.gt(0.05)).And(Frequency_all.lt(0.95)).clip(edge_roi).unmask(0).rename('mask')

    #-------------------------------------------------------------------------------------    

    Feature_band = NEF.addBands(NDWI_mean).addBands(MNDWI_mean).addBands(composit_b8A).addBands(S2_NDWI_min).addBands(TF_mask)      
                

    filename = f'{target_region}1_{year}_FB6_new9_{img_num}_{seed}'
                
    task = ee.batch.Export.image.toDrive(
        image=Feature_band.toFloat(),
        description= filename,
        folder='Discussion_51',
        fileNamePrefix=filename,
        region=roi,
        scale=30,
        maxPixels=1e13
        )
    task.start()
    print(f"\n✅ {filename}已提交，请到 https://code.earthengine.google.com/tasks 查看进度")


✅ Ribble1_2020_FB6_new9_10_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Ribble1_2020_FB6_new9_20_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Ribble1_2020_FB6_new9_30_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Ribble1_2020_FB6_new9_40_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Ribble1_2020_FB6_new9_50_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Ribble1_2020_FB6_new9_60_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Ribble1_2020_FB6_new9_70_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Ribble1_2020_FB6_new9_79_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度


In [18]:

#----------------for batch region---------------------
# target_region = 'Wash_bay'
# year = 2022
# img_num_list = [10, 20, 30, 40, 50, 60, 70, 80, 90, 93]
# img_num2_list = [10, 20, 30, 40, 50, 60, 68, 68, 68, 68]

# target_region = 'Ribble'
# year = 2020
# img_num_list = [10, 20, 30, 40, 50, 60, 70, 79]
# img_num2_list = [10, 20, 30, 40, 50, 60, 66, 66]
# seed = 5


target_region = 'Morecambe'
year = 2023
img_num_list = [10, 20, 30, 40, 50, 60, 70, 76]
img_num2_list = [10, 20, 30, 40, 46, 46, 46, 46]
seed = 5

for i in range(len(img_num_list)):
    
    img_num  = img_num_list[i]
    img_num2 = img_num2_list[i]


    start_date, end_date = f'{year}-01-01', f'{year+1}-01-01'

    # --- 选 Morecambe 区 ---
    path = r"E:\GEEdownload\UK_TF_roi\UK_FK.shp"
    gdf = gpd.read_file(path)
    region_grid = gdf[(gdf['name'] == target_region)]
    MGRS_TILE = region_grid['MGRS_TILE'].values[0]
    # print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

    if region_grid.empty:
        raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")
    merged_ = unary_union([force_2d(geom) for geom in region_grid.geometry if geom is not None])
    roi = shapely_to_ee_geometry(merged_)


    edge_path = f'E:\GEEdownload\TK-TF_edge_collect\{target_region}_edge.shp'
    edge_gdf = gpd.read_file(edge_path)
    edge_gdf  = geemap.geopandas_to_ee(edge_gdf)       
    edge_roi = edge_gdf.geometry() 

    cloud_limit = 40

    def S2_procession(collection):
    
        collection_c = collection.map(F.Cal_doy).map(F.add_cloud_bands)
                                    
        doy_list = collection_c.aggregate_array('DOY').distinct()
        
        def doy_(doy):
            
            DOY = ee.Number(doy)
            unique_doy = collection_c.filter(ee.Filter.eq('DOY', DOY)).mosaic().set('DOY', DOY)
            return unique_doy

        unique_doy_colect = doy_list.map(doy_)
        
        return ee.ImageCollection(unique_doy_colect).map(CLIP).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40))
        


    S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 100)

    if MGRS_TILE != 'False':
        print('--apply MGRS_TILE--', MGRS_TILE)
        S2 = S2.filter(ee.Filter.eq('MGRS_TILE', '30UYC'))
      
    S2 = S2_procession(S2).map(F.VI_cal_S2)
    # S2 = S2.map(F.VI_cal_S2).map(F.add_cloud_bands)\
    #     .map(CLIP).map(cal_cloud_per)\
    #     .filter(ee.Filter.gt('cloud_per', cloud_limit))

    L9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    L8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    L9 = F.Landsat_processing(roi, L9_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)
    L8 = F.Landsat_processing(roi, L8_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)


    S2 = S2.randomColumn('random', seed).sort('random')
    S2 = ee.ImageCollection(S2)
    S2_ = S2.map(F.normalized_TFI).map(F.TF_get)


    S2_TF = S2_.map(F.TF_binary_cal).toBands().select(ee.List.sequence(0, img_num2-1))
    S2_cloud = S2_.select('cloud').toBands().select(ee.List.sequence(0, img_num2-1))

    S2_sum_TF = S2_TF.reduce(ee.Reducer.sum())
    S2_cloud_sum = S2_cloud.reduce(ee.Reducer.sum())

    NEF = S2_sum_TF.divide(S2_cloud_sum).rename('NEF')
    NDWI_mean = S2_.select('NDWI').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).rename('mean_NDWI')
    MNDWI_mean = S2_.select('MNDWI').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).rename('mean_MNDWI')
    composit_b8A = S2_.select('B8A').toBands().select(ee.List.sequence(0, img_num2-1)).reduce(ee.Reducer.sum()).divide(S2_cloud_sum).divide(10000).rename('mean_B8A')


    if img_num > img_num2:#加入 L8/L9 影像
        
        L89_num = img_num - img_num2
        L89_TF_collect = L8.merge(L9).randomColumn('random', seed).sort('random')
        L89_TF_collect = ee.ImageCollection(L89_TF_collect)
        L89_sum_TF = L89_TF_collect.map(F.TF_binary_cal).toBands().select(ee.List.sequence(0, L89_num-1))#.reduce(ee.Reducer.sum())
        L89_sum_cloud = L89_TF_collect.select('cloud').toBands().select(ee.List.sequence(0, L89_num-1))#.reduce(ee.Reducer.sum())
        
        sum_TF = L89_sum_TF.addBands(S2_TF).reduce(ee.Reducer.sum())
        sum_cloud = L89_sum_cloud.addBands(S2_cloud).reduce(ee.Reducer.sum())
        NEF = sum_TF.divide(sum_cloud).rename('NEF')
        
    #-------------apply all images to construct mask and S2_NDWI_min---------------------

    all_TF_collect = S2.merge(L8).merge(L9)
    all_TF_collect = ee.ImageCollection(all_TF_collect)
    sum_TF = all_TF_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
    sum_cloud = all_TF_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
    Frequency_all = sum_TF.divide(sum_cloud).rename('Freq')

    S2_2022 = S2.map(F.normalized_TFI).map(F.TF_get)
    S2_NDWI_min = S2_2022.select('NDWI').reduce(ee.Reducer.percentile([2])).rename('S2_NDWI_min')

    TF_mask = S2_NDWI_min.lt(-0.1).And(Frequency_all.gt(0.05)).And(Frequency_all.lt(0.95)).clip(edge_roi).unmask(0).rename('mask')

    #-------------------------------------------------------------------------------------    

    Feature_band = NEF.addBands(NDWI_mean).addBands(MNDWI_mean).addBands(composit_b8A).addBands(S2_NDWI_min).addBands(TF_mask)      
                

    filename = f'{target_region}_{year}_FB6_new9_{img_num}_{seed}'
                
    task = ee.batch.Export.image.toDrive(
        image=Feature_band.toFloat(),
        description= filename,
        folder='Discussion_51',
        fileNamePrefix=filename,
        region=roi,
        scale=30,
        maxPixels=1e13
        )
    task.start()
    print(f"\n✅ {filename}已提交，请到 https://code.earthengine.google.com/tasks 查看进度")


✅ Morecambe_2023_FB6_new9_10_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Morecambe_2023_FB6_new9_20_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Morecambe_2023_FB6_new9_30_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Morecambe_2023_FB6_new9_40_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Morecambe_2023_FB6_new9_50_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Morecambe_2023_FB6_new9_60_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Morecambe_2023_FB6_new9_70_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ Morecambe_2023_FB6_new9_76_5已提交，请到 https://code.earthengine.google.com/tasks 查看进度
